In [0]:
MY_ID = "2328073"

CATALOG = "workspace"
SCHEMA = "dbsf_2328073"
VOL = "/Volumes/workspace/dbsf_2328073/raw_files"

print("MY_ID:", MY_ID)
print("SCHEMA:", SCHEMA)
print("VOL:", VOL)

MY_ID: 2328073
SCHEMA: dbsf_2328073
VOL: /Volumes/workspace/dbsf_2328073/raw_files


In [0]:
from pyspark.sql import functions as F

# ============================================================
# LOAD SILVER TABLES
# ============================================================

orders = spark.table(
    "workspace.dbsf_2328073.silver_order_metrics_2328073"
)

order_category = spark.table(
    "workspace.dbsf_2328073.silver_order_category_2328073"
)

seller_order = spark.table(
    "workspace.dbsf_2328073.silver_seller_order_2328073"
)

sellers = spark.table(
    "workspace.dbsf_2328073.silver_sellers_2328073"
)

print("Silver tables loaded.")

Silver tables loaded.


In [0]:
# ============================================================
# GOLD 1
# CATEGORY DELIVERY & REVIEW ANALYSIS
# ============================================================

category_data = (
    order_category

    .join(
        orders.select(
            "order_id",
            "order_status",
            "delivery_days",
            "late_flag",
            "avg_review_score"
        ),
        on="order_id",
        how="inner"
    )

    # Only delivered orders can have delivery performance
    .filter(
        (F.col("order_status") == "delivered")
        & F.col("delivery_days").isNotNull()
    )
)


gold_category = (
    category_data

    .groupBy("category")

    .agg(
        F.countDistinct("order_id")
            .alias("orders"),

        F.round(
            F.sum("category_revenue"),
            2
        ).alias("revenue"),

        F.round(
            F.avg("delivery_days"),
            2
        ).alias("avg_delivery_days"),

        F.sum("late_flag")
            .alias("late_orders"),

        F.round(
            F.avg("avg_review_score"),
            2
        ).alias("avg_review_score"),

        F.round(
            F.avg(
                F.when(
                    F.col("late_flag") == 1,
                    F.col("avg_review_score")
                )
            ),
            2
        ).alias("late_avg_review_score"),

        F.round(
            F.avg(
                F.when(
                    F.col("late_flag") == 0,
                    F.col("avg_review_score")
                )
            ),
            2
        ).alias("on_time_avg_review_score")
    )

    .withColumn(
        "late_rate",
        F.round(
            F.col("late_orders")
            / F.col("orders"),
            4
        )
    )

    .withColumn(
        "review_score_gap",
        F.round(
            F.col("on_time_avg_review_score")
            - F.col("late_avg_review_score"),
            2
        )
    )
)


gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.gold_category_delivery_review_2328073"
    )


print(
    "Gold category rows:",
    gold_category.count()
)

Gold category rows: 74


In [0]:
# ============================================================
# GOLD 2
# REPEAT CUSTOMER REVENUE BY QUARTER
# ============================================================

# Find customers with more than one order
repeat_customers = (
    orders

    .filter(
        F.col("customer_unique_id").isNotNull()
    )

    .groupBy("customer_unique_id")

    .agg(
        F.countDistinct("order_id")
            .alias("order_count")
    )

    .filter(
        F.col("order_count") > 1
    )

    .select(
        "customer_unique_id"
    )
)


# Add repeat-customer flag
order_revenue = (
    orders

    .filter(
        F.col("purchase_ts").isNotNull()
    )

    .select(
        "order_id",
        "purchase_ts",
        "customer_unique_id",
        "order_revenue"
    )
)


order_revenue = (
    order_revenue

    .join(
        repeat_customers
        .withColumn(
            "repeat_customer",
            F.lit(1)
        ),
        on="customer_unique_id",
        how="left"
    )

    .fillna(
        {"repeat_customer": 0}
    )
)


# Create quarter
order_revenue = order_revenue.withColumn(
    "year",
    F.year("purchase_ts")
)


order_revenue = order_revenue.withColumn(
    "quarter_number",
    F.quarter("purchase_ts")
)


order_revenue = order_revenue.withColumn(
    "quarter",
    F.concat(
        F.col("year").cast("string"),
        F.lit("-Q"),
        F.col("quarter_number").cast("string")
    )
)


# Aggregate
gold_repeat = (
    order_revenue

    .groupBy("quarter")

    .agg(
        F.countDistinct("order_id")
            .alias("total_orders"),

        F.round(
            F.sum("order_revenue"),
            2
        ).alias("total_revenue"),

        F.sum(
            F.when(
                F.col("repeat_customer") == 1,
                1
            ).otherwise(0)
        ).alias("repeat_orders"),

        F.round(
            F.sum(
                F.when(
                    F.col("repeat_customer") == 1,
                    F.col("order_revenue")
                ).otherwise(0)
            ),
            2
        ).alias("repeat_revenue")
    )

    .withColumn(
        "repeat_revenue_share",
        F.round(
            F.col("repeat_revenue")
            / F.col("total_revenue"),
            4
        )
    )
)


gold_repeat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.gold_repeat_customer_quarter_2328073"
    )


print(
    "Gold repeat-customer rows:",
    gold_repeat.count()
)

Gold repeat-customer rows: 10


In [0]:
# ============================================================
# GOLD 3
# SELLER PERFORMANCE
# ============================================================

gold_seller = (
    seller_order

    .filter(
        F.col("order_status") == "delivered"
    )

    .groupBy("seller_id")

    .agg(
        F.countDistinct("order_id")
            .alias("delivered_orders"),

        F.sum("late_flag")
            .alias("late_orders"),

        F.round(
            F.sum("seller_revenue"),
            2
        ).alias("revenue"),

        F.round(
            F.avg("avg_review_score"),
            2
        ).alias("avg_review_score")
    )

    .withColumn(
        "late_rate",
        F.round(
            F.col("late_orders")
            / F.col("delivered_orders"),
            4
        )
    )

    .join(
        sellers.select(
            "seller_id",
            "seller_city",
            "seller_state"
        ),
        on="seller_id",
        how="left"
    )
)


gold_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.gold_seller_performance_2328073"
    )


print(
    "Gold seller rows:",
    gold_seller.count()
)

Gold seller rows: 2970


In [0]:
gold_tables = [
    "gold_category_delivery_review_2328073",
    "gold_repeat_customer_quarter_2328073",
    "gold_seller_performance_2328073"
]

print("======================================")
print("GOLD TABLE VALIDATION")
print("======================================")

for table in gold_tables:

    full_name = f"workspace.dbsf_2328073.{table}"

    print(
        f"{table:<50}",
        f"{spark.table(full_name).count():,}"
    )

GOLD TABLE VALIDATION
gold_category_delivery_review_2328073              74
gold_repeat_customer_quarter_2328073               10
gold_seller_performance_2328073                    2,970


In [0]:
display(
    spark.table(
        "workspace.dbsf_2328073.gold_category_delivery_review_2328073"
    )
    .orderBy(
        F.col("late_rate").desc()
    )
    .limit(20)
)

category,orders,revenue,avg_delivery_days,late_orders,avg_review_score,late_avg_review_score,on_time_avg_review_score,late_rate,review_score_gap
home_comfort_2,24,760.27,14.46,4,3.83,1.75,4.26,0.1667,2.51
furniture_mattress_and_upholstery,37,4323.38,14.41,5,3.89,2.2,4.16,0.1351,1.96
audio,348,50570.6,13.42,45,3.84,1.98,4.11,0.1293,2.13
fashion_underwear_beach,117,9305.95,13.91,15,4.01,2.07,4.3,0.1282,2.23
books_technical,256,18702.23,10.55,28,4.43,3.61,4.53,0.1094,0.92
home_confort,392,58008.45,13.64,41,3.89,2.49,4.05,0.1046,1.56
food,441,28731.15,9.73,44,4.33,3.23,4.45,0.0998,1.22
electronics,2517,155043.93,12.84,247,4.12,2.75,4.27,0.0981,1.52
christmas_supplies,125,8737.84,15.3,12,4.11,2.73,4.25,0.096,1.52
baby,2809,400421.84,12.61,258,4.1,2.33,4.28,0.0918,1.95


In [0]:
display(
    spark.table(
        "workspace.dbsf_2328073.gold_repeat_customer_quarter_2328073"
    )
    .orderBy("quarter")
)

quarter,total_orders,total_revenue,repeat_orders,repeat_revenue,repeat_revenue_share
2016-Q3,4,267.36,0,0.0,0.0
2016-Q4,325,49518.56,16,1670.34,0.0337
2017-Q1,5262,741960.19,371,41116.8,0.0554
2017-Q2,9349,1299036.97,741,94368.34,0.0726
2017-Q3,12642,1696404.85,1007,122446.25,0.0722
2017-Q4,17848,2418404.97,1146,138617.78,0.0573
2018-Q1,21208,2777422.51,1403,164033.73,0.0591
2018-Q2,19979,2858289.74,1027,136380.13,0.0477
2018-Q3,12820,1750338.55,628,80188.6,0.0458
2018-Q4,4,null,3,0.0,null


In [0]:
display(
    spark.table(
        "workspace.dbsf_2328073.gold_seller_performance_2328073"
    )
    .orderBy(
        F.col("late_rate").desc()
    )
    .limit(20)
)

seller_id,delivered_orders,late_orders,revenue,avg_review_score,late_rate,seller_city,seller_state
f5fea3ffed6c2e889bab72705557c63a,1,1,345.0,5.0,1.0,rio de janeiro,RJ
1f2eebc0e970fd3c463e4f5d9652687a,1,1,44.99,5.0,1.0,sao paulo,SP
be1e9e378700cecaa4ebf71433d7915c,2,2,119.89,3.0,1.0,ribeirao preto,SP
19484c79cef6c062cb177aa4ef2fcc3c,1,1,201.9,1.0,1.0,valinhos,SP
535bf14f36346c7f59d87cece104d70c,1,1,50.0,4.0,1.0,rio de janeiro,RJ
ca5832c6960267b71041f74bb39e8b12,1,1,79.9,1.0,1.0,xanxere,SC
791cfcfe22fe4a771ece27f90017da92,1,1,298.13,1.0,1.0,ribeirao preto,SP
51a04a8a6bdcb23deccc82b0b80742cf,1,1,167.99,1.0,1.0,braganca paulista,SP
da2782c804606d2a5d8e1760dbb3e7ec,1,1,89.0,1.0,1.0,sao paulo,SP
2a50b7ee5aebecc6fd0ff9784a4747d6,1,1,199.8,1.0,1.0,brasilia,DF


In [0]:
# ============================================================
# EXPORT GOLD TABLES FOR SNOWFLAKE
# ============================================================

EXPORT_BASE = f"{VOL}/gold_export"

exports = {
    "category_delivery_review":
        "workspace.dbsf_2328073.gold_category_delivery_review_2328073",

    "repeat_customer_quarter":
        "workspace.dbsf_2328073.gold_repeat_customer_quarter_2328073",

    "seller_performance":
        "workspace.dbsf_2328073.gold_seller_performance_2328073"
}


for name, table_name in exports.items():

    export_path = f"{EXPORT_BASE}/{name}"

    print(f"Exporting {name}...")

    (
        spark.table(table_name)
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(export_path)
    )

    print(f"Done: {export_path}")


print("\n================================")
print("GOLD EXPORT COMPLETE")
print("================================")

Exporting category_delivery_review...
Done: /Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review
Exporting repeat_customer_quarter...
Done: /Volumes/workspace/dbsf_2328073/raw_files/gold_export/repeat_customer_quarter
Exporting seller_performance...
Done: /Volumes/workspace/dbsf_2328073/raw_files/gold_export/seller_performance

GOLD EXPORT COMPLETE


In [0]:
display(
    dbutils.fs.ls(
        f"{VOL}/gold_export"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review/,category_delivery_review/,0,1790745518711
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/gold_category_delay_summary/,gold_category_delay_summary/,0,1790745518711
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/gold_customer_repeat_behavior/,gold_customer_repeat_behavior/,0,1790745518711
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/gold_seller_delay_summary/,gold_seller_delay_summary/,0,1790745518711
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/repeat_customer_quarter/,repeat_customer_quarter/,0,1790745518711
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/seller_performance/,seller_performance/,0,1790745518711


In [0]:
display(
    dbutils.fs.ls(
        f"{VOL}/gold_export/category_delivery_review"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review/_SUCCESS,_SUCCESS,0,1790745428000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review/_committed_2843880466675091467,_committed_2843880466675091467,113,1790745428000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review/_started_2843880466675091467,_started_2843880466675091467,0,1790745427000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/category_delivery_review/part-00000-tid-2843880466675091467-7e404510-8dce-4ac8-aead-e444fab0c3a3-427-1-c000.csv,part-00000-tid-2843880466675091467-7e404510-8dce-4ac8-aead-e444fab0c3a3-427-1-c000.csv,4909,1790745428000
